# Etapa 3 - MapReduce con Spark RDDs

En este notebook resolvemos cuatro preguntas de negocio del dominio de la fabrica de pastas usando la API de RDDs de PySpark. En cada caso se detalla que ocurre en la fase de **Map**, que sucede en la fase de **Reduce** y cuando Spark ejecuta efectivamente el computo.

Los datos utilizados provienen de los archivos CSV generados a partir de la Etapa 1.

## Consultas analizadas

1. ¿Qué ingredientes aparecen en más rellenos distintos?
2. ¿Cuál es el gasto promedio por compra en cada franquicia?
3. ¿Cuáles son las 10 pastas más vendidas por kilos?
4. ¿Qué meses concentran la mayor cantidad de compras?


## Requisitos

- Tener instalado `pyspark`.
- Tener Java 11 o 17 disponible en el sistema.
- Ejecutar este notebook desde la carpeta del repositorio.

In [55]:
##  %pip install pyspark

from pyspark import SparkContext
try:
    sc.stop()
except:
    pass

In [56]:
import os
from pyspark import SparkConf, SparkContext
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

conf = (SparkConf()
        .setMaster("local[*]")        # Corre Spark en tu máquina local, usando todos los núcleos disponibles del CPU ("local" a secas, corre Spark en modo local con un solo hilo (1 CPU))
        .setAppName("Grupo 11 - TP Etapa 3")
        .set("spark.driver.bindAddress", "127.0.0.1")
        .set("spark.driver.host", "127.0.0.1"))
sc = SparkContext(conf = conf)
sc

<SparkContext master=local[*] appName=Grupo 11 - TP Etapa 3>

In [57]:
def load_csv_as_dict_rdd(path):
    raw_rdd = sc.textFile(str(path))
    header = raw_rdd.first()
    headers = next(csv.reader([header]))

    def parse_line(line):
        values = next(csv.reader([line]))
        return dict(zip(headers, values))

    return raw_rdd.filter(lambda line: line != header).map(parse_line)

In [58]:
from pathlib import Path
dataPath = Path("../data")

ingrediente_rdd = load_csv_as_dict_rdd(dataPath / "ingrediente.csv")
relleno_ingrediente_rdd = load_csv_as_dict_rdd(dataPath / "relleno_ingrediente.csv")
cliente_rdd = load_csv_as_dict_rdd(dataPath / "cliente.csv")
compra_rdd = load_csv_as_dict_rdd(dataPath / "compra.csv")
detalle_compra_rdd = load_csv_as_dict_rdd(dataPath / "detalle_compra.csv")
pasta_rdd = load_csv_as_dict_rdd(dataPath / "pasta.csv")


In [59]:
def print_rows(rows, title=None):
    if title:
        print(title)
    for row in rows:
        print(row)

## Procesamiento 1
### ¿Qué ingredientes aparecen en más rellenos distintos?

**Pregunta de negocio.** Este procesamiento busca detectar que ingredientes son los más versatiles dentro del catalogo, es decir, cuales se reutilizan en una mayor cantidad de rellenos diferentes.

**Fase de Map.** A partir de `relleno_ingrediente.csv`, cada fila se transforma primero en la clave compuesta `((id_ingrediente, id_relleno), 1)` para identificar cada combinacion ingrediente-relleno. Luego, una vez eliminados posibles duplicados con `distinct()`, se vuelve a mapear a `(id_ingrediente, 1)`.

**Fase de Reduce.** Se aplica `reduceByKey()` sobre `id_ingrediente` para sumar cuántos rellenos distintos tiene asociado cada ingrediente.

**Resultado final.** Se reemplaza el identificador por el nombre del ingrediente, se ordena por cantidad y se usa `top(10)` para mostrar los diez ingredientes que participan en más rellenos.


In [60]:
ingredientes_dict = dict(
    ingrediente_rdd
    .map(lambda row: (row["id_ingrediente"], row["nombre"]))
    .collect()
)

top_ingredientes = (
    relleno_ingrediente_rdd
    .map(lambda row: ((row["id_ingrediente"], row["id_relleno"]), 1))  # identifica ingrediente-relleno
    .distinct()  # evita contar repetidos dentro del mismo relleno
    .map(lambda par: (par[0][0], 1))  # deja solo el ingrediente
    .reduceByKey(lambda a, b: a + b)  # cuenta en cuantos rellenos distintos aparece
    .map(lambda par: (par[1], ingredientes_dict[par[0]])) # reemplaza id por nombre y cambio el orden para que la cantidad quede primero (para ordenar)
    .sortByKey() # ordeno por cantidad
    .top(10) # agarro el top 10
)

print_rows(top_ingredientes, "Ingredientes que participan en más rellenos distintos:")


Ingredientes que participan en más rellenos distintos:
(11, 'Choclo')
(10, 'Gorgonzola')
(8, 'Ajo')
(7, 'Trufa')
(7, 'Hongos')
(7, 'Berenjena')
(6, 'Zucchini')
(6, 'Ricota')
(6, 'Perejil')
(6, 'Nuez moscada')


## Procesamiento 2
### ¿Cuál es el gasto promedio por compra en cada franquicia?

**Pregunta de negocio.** Acá queremos comparar el comportamiento de consumo entre franquicias, midiendo cuánto se gasta en promedio cada vez que se realiza una compra.

**Fase de Map.**

1. En `detalle_compra.csv`, cada linea se transforma en `(id_compra, subtotal)`, donde `subtotal = cantidad_kg * precio_unitario`.
2. En `compra.csv`, cada compra se representa como `(id_compra, id_cliente)`.
3. En `cliente.csv`, cada cliente se representa como `(id_cliente, sello)`.

**Fase de Reduce.**

1. Se suman los subtotales por `id_compra` para obtener el monto total de cada compra.
2. Mediante joins, ese total se vincula primero con el cliente y luego con la franquicia correspondiente.
3. Ya con la clave `sello`, se agrupan los datos como `(monto_total_acumulado, cantidad_de_compras)`.
4. Finalmente, para cada franquicia se calcula el promedio dividiendo el monto acumulado por la cantidad de compras.

**Resultado final.** Se ordenan los promedios de menor a mayor y se usa `collect()` para traer el resultado final y mostrarlo en pantalla.


In [61]:
subtotal_por_compra = (
    detalle_compra_rdd
    .map(lambda row: (row["id_compra"], float(row["cantidad_kg"]) * float(row["precio_unitario"])))  # calcula el subtotal de cada detalle de compra
    .reduceByKey(lambda a, b: a + b)  # suma los subtotales para obtener el total de cada compra
)

cliente_por_compra = (
    compra_rdd
    .map(lambda row: (row["id_compra"], row["id_cliente"]))  # relaciona cada compra con su cliente
)

sello_por_cliente = (
    cliente_rdd
    .map(lambda row: (row["id_cliente"], row["sello"]))  # relaciona cada cliente con su franquicia/sello
)

promedio_gasto_por_franquicia = (
    subtotal_por_compra
    .join(cliente_por_compra)  # une el total de la compra con el cliente que la realizada
    .map(lambda item: (item[1][1], item[1][0]))  # reordena para dejar como clave el id_cliente y como valor el subtotal de la compra
    .join(sello_por_cliente)  # une cada cliente con su franquicia
    .map(lambda item: (item[1][1], (item[1][0], 1)))  # deja como clave la franquicia y como valor (monto_compra, 1) para luego calcular promedio
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))  # suma montos y cantidad de compras por franquicia
    .map(lambda item: (item[0], round(item[1][0] / item[1][1], 2)))  # calcula el promedio de gasto por compra en cada franquicia
    .sortBy(lambda item: (item[1], item[0])) # ordeno de menor a mayor
    .collect()
)


print_rows(promedio_gasto_por_franquicia, "Promedio de gasto por compra en cada franquicia:")


Promedio de gasto por compra en cada franquicia:
('FR-004', 23502.77)
('FR-014', 24475.65)
('FR-005', 27112.2)
('FR-007', 27398.73)
('FR-003', 28012.18)
('FR-010', 29109.21)
('FR-001', 29144.95)
('FR-002', 29188.83)
('FR-011', 29342.75)
('FR-013', 29885.38)
('FR-008', 30917.64)
('FR-015', 31347.84)
('FR-006', 32681.63)
('FR-009', 33245.21)
('FR-012', 34006.71)


## Procesamiento 3
### ¿Cuáles son las 10 pastas más vendidas por kilos?

**Pregunta de negocio.** El objetivo es identificar que productos concentran el mayor volumen de ventas, medido en kilos vendidos.

**Fase de Map.**

1. Desde `detalle_compra.csv` se genera `((sello, codigo_pasta), cantidad_kg)` para registrar cuántos kilos se vendieron de cada pasta dentro de cada franquicia.
2. Desde `pasta.csv` se genera `((sello, codigo_pasta), nombre_pasta)` para poder traducir luego el codigo al nombre comercial.

**Fase de Reduce.**

1. Se suman los kilos por la clave compuesta `(sello, codigo_pasta)`.
2. Se hace un join con el RDD de pastas para recuperar el nombre.
3. Como una misma pasta puede existir en más de una franquicia, se vuelve a agrupar por `nombre_pasta` y se suman los kilos totales.

**Resultado final.** Se ordena de mayor a menor segun la cantidad de kilos vendidos y se usa `take(10)` para obtener el ranking final.


In [62]:
kg_por_pasta_y_sello = (
    detalle_compra_rdd
    .map(lambda row: ((row["sello"], row["codigo_pasta"]), float(row["cantidad_kg"])))  # toma cada venta y deja como clave (sello, pasta) y como valor los kg vendidos
    .reduceByKey(lambda a, b: a + b)  # suma los kilos vendidos por cada pasta dentro de cada sello
)

nombre_por_pasta_y_sello = (
    pasta_rdd
    .map(lambda row: ((row["sello"], row["codigo_pasta"]), row["nombre"]))  # relaciona cada pasta de cada sello con su nombre
)

top_pastas_por_kg = (
    kg_por_pasta_y_sello
    .join(nombre_por_pasta_y_sello)  # une los kilos vendidos con el nombre de la pasta
    .map(lambda item: (item[1][1], item[1][0]))  # deja como clave el nombre de la pasta y como valor los kg vendidos
    .reduceByKey(lambda a, b: a + b)  # suma los kilos si una misma pasta aparece en mas de un sello
    .sortBy(lambda item: (-item[1], item[0])) # ordena de mayor a menor
    .take(10)  # toma las 10 pastas con mas kilos vendidos
)

print_rows(top_pastas_por_kg, "Top 10 pastas por kilos vendidos:")


Top 10 pastas por kilos vendidos:
('Panzotti', 12332.101000000002)
('Capeletti', 11562.827000000001)
('Ravioles', 11040.314999999999)
('Tagliatelle', 9665.613000000001)
('Ravioloni', 9629.443)
('Spaghetti', 9307.559)
('Lasaña', 9174.102)
('Agnolotti', 8925.318)
('Tortellini', 8683.305)
('Sorrentinos', 8202.497)


## Procesamiento 4
### ¿Qué meses concentran la mayor cantidad de compras?

**Pregunta de negocio.** Este analisis permite detectar picos de demanda, observando en que meses se registro la mayor cantidad de compras.

**Fase de Map.** A partir de `compra.csv`, cada compra se transforma en el par `(MM, 1)`, tomando de `fecha_hora` solamente el mes.

**Fase de Reduce.** Se aplica `reduceByKey()` para sumar la cantidad de compras de cada mes, independiente del año.

**Resultado final.** Se ordenan los meses en forma descendente segun la cantidad de compras


In [63]:
compras_por_mes = (
    compra_rdd
    .map(lambda row: (row["fecha_hora"][5:7], 1))   # extrae solo el mes (AAAA-MM) 
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda item: (-item[1], item[0]))
)

top_meses = compras_por_mes.collect()
print_rows(top_meses, "Meses con mayor cantidad de compras:")


Meses con mayor cantidad de compras:
('10', 2440)
('03', 2362)
('01', 2361)
('05', 2333)
('04', 2307)
('11', 2303)
('06', 2267)
('12', 2262)
('09', 2251)
('07', 2233)
('08', 2227)
('02', 2113)
